In [2]:
!pip install -q ultralytics gdown
!gdown --fuzzy "https://drive.google.com/file/d/1qqY90S-MHpy2_E9-H49X7UY1VjYOGHxB/view?usp=drive_link" -O /content/yolo_medoc_v3_colab.zip
!unzip -q -o /content/yolo_medoc_v3_colab.zip -d /content
!ls /content/yolo_medoc_v3/images/train | wc -l     # should print 327

Downloading...
From (original): https://drive.google.com/uc?id=1qqY90S-MHpy2_E9-H49X7UY1VjYOGHxB
From (redirected): https://drive.google.com/uc?id=1qqY90S-MHpy2_E9-H49X7UY1VjYOGHxB&confirm=t&uuid=96d49227-c8b8-45e7-aaae-6d9708fb52b1
To: /content/yolo_medoc_v3_colab.zip
100% 36.0M/36.0M [00:00<00:00, 51.1MB/s]
327


In [3]:
from pathlib import Path
import yaml
from ultralytics import YOLO

In [4]:
# Cell 2: train yolo11s, results saved straight to Drive
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(data="/content/yolo_medoc_v3/data_colab.yaml",
            epochs=150, patience=50, imgsz=640, degrees=180,     # same as yolo11n_medoc_v3
            batch=32,                 # the T4 has 16 GB: 32 fits for yolo11s
            workers=2,                # Colab only has 2 CPUs
            cache="ram",              # 357 small images: load them once, much faster epochs
            device=0,
            project="/content/drive/MyDrive/yolo/yolo_runs", name="yolo11s_medoc_v3", exist_ok=True)

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_medoc_v3/data_colab.yaml, degrees=180, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_medoc_v3, nbs=64, nms=None, opset=None, opt

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ca107ead9b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [7]:
from huggingface_hub import HfApi, login
login(token=input("HF write token: "))     # paste the hf_... token when asked
api = HfApi()
api.create_repo("TomasAnderegg/yolo_medoc_models", private=True, exist_ok=True)
api.upload_folder(folder_path="/content/drive/MyDrive/yolo/yolo_runs/yolo11s_medoc_v3",
                  repo_id="TomasAnderegg/yolo_medoc_models", path_in_repo="yolo11s_medoc_v3")

CommitInfo(commit_url='https://huggingface.co/TomasAnderegg/yolo_medoc_models/commit/dafd733dd0cf0feac00469086fc1d3b6a6520395', commit_message='Upload folder using huggingface_hub', commit_description='', oid='dafd733dd0cf0feac00469086fc1d3b6a6520395', pr_url=None, repo_url=RepoUrl('https://huggingface.co/TomasAnderegg/yolo_medoc_models', endpoint='https://huggingface.co', repo_type='model', repo_id='TomasAnderegg/yolo_medoc_models'), pr_revision=None, pr_num=None)

In [ ]:
for run in ["yolo11n_medoc_v2", "yolo11n_medoc_v3", "yolo11s_medoc_v3"]:
    m = YOLO(f"{PROJECT}/{run}/weights/best.pt")
    r = m.val(data=DATA, verbose=False, plots=False)
    per_class = "  ".join(f"{name}={r.box.maps[i]:.3f}" for i, name in m.names.items())
    print(f"{run:18s} mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}  |  {per_class}")